In [ ]:
!pip install open_clip_torch -q

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import open_clip

device = torch.device("cpu")
torch.manual_seed(42)

In [ ]:
data_dir = "./data/dvm"

train_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_ds = datasets.ImageFolder(data_dir)
targets = [s[1] for s in full_ds.samples]
train_idx, val_idx = train_test_split(
    range(len(full_ds)), test_size=0.2, stratify=targets, random_state=42
)


class SubsetWithTransform:
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[self.indices[idx]]
        img = self.dataset.loader(path)
        if self.transform:
            img = self.transform(img)
        return img, label


train_ds = SubsetWithTransform(full_ds, train_idx, train_tfm)
val_ds = SubsetWithTransform(full_ds, val_idx, val_tfm)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

num_classes = len(full_ds.classes)
class_names = full_ds.classes
print(f"Классы ({num_classes}): {class_names}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in tqdm(range(epochs)):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += images.size(0)

        history["train_loss"].append(running_loss / total)
        history["train_acc"].append(correct / total)

        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                loss = criterion(out, labels)
                v_loss += loss.item() * images.size(0)
                v_correct += (out.argmax(1) == labels).sum().item()
                v_total += images.size(0)

        history["val_loss"].append(v_loss / v_total)
        history["val_acc"].append(v_correct / v_total)

    return history


def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            out = model(images)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)

for param in resnet.parameters():
    param.requires_grad = False
for param in resnet.fc.parameters():
    param.requires_grad = True
for param in resnet.layer4.parameters():
    param.requires_grad = True

h_resnet = train_model(resnet, train_loader, val_loader, epochs=10, lr=1e-3)

In [ ]:
clip_model, _, _ = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)

clip_train_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.48145466, 0.4578275, 0.40821073],
                         [0.26862954, 0.26130258, 0.27577711])
])
clip_val_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.48145466, 0.4578275, 0.40821073],
                         [0.26862954, 0.26130258, 0.27577711])
])

clip_train_ds = SubsetWithTransform(full_ds, train_idx, clip_train_tfm)
clip_val_ds = SubsetWithTransform(full_ds, val_idx, clip_val_tfm)
clip_train_loader = DataLoader(clip_train_ds, batch_size=32, shuffle=True, num_workers=2)
clip_val_loader = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=2)


class CLIPClassifier(nn.Module):
    def __init__(self, clip_visual, num_classes, embed_dim=512):
        super().__init__()
        self.visual = clip_visual
        for p in self.visual.parameters():
            p.requires_grad = False
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            features = self.visual(x)
        return self.head(features)


clip_clf = CLIPClassifier(clip_model.visual, num_classes)
h_clip = train_model(clip_clf, clip_train_loader, clip_val_loader, epochs=10, lr=1e-3)

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


custom_cnn = CustomCNN(num_classes)
h_custom = train_model(custom_cnn, train_loader, val_loader, epochs=20, lr=1e-3)

In [ ]:
results = {}

for name, model, loader in [
    ("ResNet18 (ImageNet)", resnet, val_loader),
    ("CLIP ViT-B/32 (LAION-2B)", clip_clf, clip_val_loader),
    ("Custom CNN", custom_cnn, val_loader),
]:
    preds, labels = get_predictions(model, loader)
    f1 = f1_score(labels, preds, average="macro")
    results[name] = f1
    print(f"\n{name}:")
    print(f"  F1_macro = {f1:.4f}")
    print(classification_report(labels, preds, target_names=class_names))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, h) in zip(axes, [
    ("ResNet18", h_resnet), ("CLIP", h_clip), ("Custom CNN", h_custom)
]):
    ax.plot(h["train_loss"], label="train")
    ax.plot(h["val_loss"], label="val")
    ax.set_title(f"{name} - Loss")
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

names = list(results.keys())
scores = list(results.values())
plt.figure(figsize=(10, 5))
plt.bar(range(len(names)), scores)
plt.xticks(range(len(names)), names, rotation=15)
plt.ylabel("F1 macro")
plt.title("Сравнение моделей по F1 macro")
plt.grid(axis="y")
plt.tight_layout()
plt.show()